# UCB-FE Experiments

This notebooks provides all neccessary calculations for the UCB-FE experiments.
Logic about using ML models are implemeted in the [utils file](./utils_ucb_fe.py).

Note that you can either use pretrained models with weights we provided or train all models by your self. The whole training process will take approximately 6 hours on GPU (all models), calculating metrics on CPU will take nearly 1 hour (all models).

P.S. It is possible to verify the code by excluding the tabm model from the calculations, reducing the model training time to 1 hour.

In [1]:
import numpy  as np
import pandas as pd

In [ ]:
# read log data from the 1.1 UCB-FE-exp
df =  pd.read_parquet('data/results/df_ucb_fe.parquet')

In [3]:
# params
model_names = ['catboost','lightgbm', 'xgboost', 'tabnet']
cold_periods = [0, 10, 100, 200, 500]
top_n = 10
tail_m = 30

In [4]:
def custom_metrics(df, model_names, cold_periods, n, m, lam):


    """
    ----------------------------------------------------------------------------------------
    model_names : predefined list of names of pre-trained models ([catboost, xgboost,...])
    cold_periods : list of values of cold-start periods ([0, 10, 100,...])
    n : lower level of top positions (...8, 9, 10)
    m : upper level of tail positions (30, 31, 32....)
    lam : basis of the degree for the position (0.98)
    ----------------------------------------------------------------------------------------

    Сalculate the necessary values for calculating the 'clicks' metrics such as CBIQ in the chapter 3.4.3
    
    """    

    ad_df = pd.DataFrame()
    for name in model_names:
        for T in cold_periods:
        
            with_click_in_tail_m = df[(df['target'] > 0) & (df['pos_old_' + name] >= m)].shape[0]
            with_click_from_tail_m_to_top_n = df[(df['target'] > 0)  & (df['pos_old_' + name] >= m) & (df['pos_new_T_'+str(T) + '_' + name] < n)].shape[0]
            with_click_in_top_m = df[(df['target'] > 0) & (df['pos_old_' + name] < m)].shape[0]
            with_click_from_top_n_to_tail_m = df[(df['target'] > 0)  & (df['pos_old_' + name] < n) & (df['pos_new_T_'+str(T) + '_' + name] >= m)].shape[0]

            imps_cold_base=  np.sum(lam ** df['pos_old_' + name] * df['cold_'+str(T)])
            imps_cold_ucb =  np.sum(lam ** df['pos_new_T_'+str(T) + '_' + name] * df['cold_'+str(T)])

            clicks_cold_base = np.sum(lam ** df['pos_old_' + name] * df['cold_'+str(T)] * df['target'])
            clicks_cold_ucb = np.sum(lam ** df['pos_new_T_'+str(T) + '_' + name] * df['cold_'+str(T)] * df['target'])

            clicks_base = np.sum(lam ** df['pos_old_' + name] * df['target'])
            clicks_ucb =  np.sum(lam ** df['pos_new_T_'+str(T) + '_' + name] * df['target'])

    
            df_result= pd.DataFrame({'model': [name], 'cold_period' : [T],
                                    'CBIQ' :  [100*with_click_from_tail_m_to_top_n/with_click_in_tail_m],
                                    'CBRQ' : [100*with_click_from_top_n_to_tail_m/with_click_in_top_m],
                                    'IL_@' + str(lam) + ' Baseline' : [imps_cold_base], 
                                    'IL_@' + str(lam) +' FE-UCB'  : [imps_cold_ucb], 
                                    'coldCL@' + str(lam) +' Baseline' : [clicks_cold_base], 
                                    'coldCL@' + str(lam) +' FE-UCB' : [clicks_cold_ucb],
                                    'CL_@' + str(lam) +' Baseline' : [clicks_base],
                                    'CL_@' + str(lam) +' FE-UCB' : [clicks_ucb],
                                  })
    
            ad_df = pd.concat([ad_df, df_result])

    return ad_df


In [5]:
result = custom_metrics(df, model_names, cold_periods, n = top_n, m = tail_m, lam = 0.98)
result.to_csv('data/results/custom_metrics_0.98.csv', index = False)
result

KeyError: 'pos_old_lightgbm'

In [ ]:
result = custom_metrics(df, model_names, cold_periods, n = top_n, m = tail_m, lam = 0.9)
result.to_csv('data/results/custom_metrics_0.90.csv', index = False)
result

In [ ]:
result = custom_metrics(df, model_names, cold_periods, n = top_n, m = tail_m, lam = 0.8)
result.to_csv('data/results/custom_metrics_0.90.csv', index = False)
result